# Thresholds and the fallback route [Step 05.03 - Knowing when you don't know]

> **MLCourse - Agentic AI - Agent Patterns**

A router always returns a best route. The threshold is what turns "best" into
"good enough", and the **fallback route** is what happens when it is not.

This is the most important notebook in the module. Thresholds are where routing
either saves you money or quietly ruins your answers.

### What you'll learn

- What the score distribution actually looks like for in-domain vs out-of-domain
  traffic, measured.
- Sweeping the threshold and reading the **coverage vs accuracy** curve.
- Why the two error types are not symmetric, and how that decides your threshold.
- The **fallback route** as a first-class design element, not an else-branch.

### Key takeaways

- Pick the threshold from a curve you measured, never from a number someone posted
  online. It depends on your encoder, your routes and your traffic.
- **Missed-route errors are cheap** (you pay for a model call you could have
  avoided). **Misroute errors are expensive** (the user gets a confidently wrong
  answer from the wrong handler). Bias the threshold high.
- The fallback path must be genuinely capable, because everything unusual goes there.

### Setup: environment, model factory, rate-limit-aware call helper


In [ ]:
import os                                   # environment variables
import time                                 # timing + backoff sleeps
from pathlib import Path                    # locating the .env
from dotenv import load_dotenv              # reads KEY=value pairs from .env

# Walk UP from this notebook until we find the folder that CONTAINS the track
# directory `03_agentic_ai` (that folder is the repo root), then load the
# gitignored .env that lives INSIDE the track.
#
# Pitfall worth naming: it is easy to write the walk so that it stops at the
# repo root and then load `ROOT/.env`, which does not exist - `load_dotenv`
# returns False and says nothing, so the notebook silently has no key.
ROOT = Path.cwd()
while not (ROOT / "03_agentic_ai").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
ENV_PATH = ROOT / "03_agentic_ai" / ".env"
load_dotenv(ENV_PATH)

GROQ_MODEL = "qwen/qwen3.8-27b"             # the one hosted model this course uses
GROQ_KEY = os.environ["GROQ_API_KEY"]       # KeyError here = .env not found. Never print it.

# A local Ollama model (e.g. `llama3.1:8b`) is a perfectly good substitute if you
# have no Groq key - swap the two lines in `make_llm`. We deliberately do NOT
# write a silent fallback branch: a notebook that quietly changes model behind
# your back produces numbers you cannot trust.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 256):
    """Return the chat model used everywhere in this module."""
    return ChatGroq(model=GROQ_MODEL, api_key=GROQ_KEY,
                    temperature=temperature, max_tokens=max_tokens)


PACE = 0.7          # seconds to wait between calls: the free tier is 8000 TPM


def safe_invoke(model, messages, retries: int = 5, pause: float = 2.0):
    """Invoke a chat model, backing off exponentially on 429 / rate-limit errors.

    Returns the AIMessage. Raises if every retry is exhausted - we want a loud
    failure, not a quiet wrong number.
    """
    for attempt in range(retries):
        try:
            out = model.invoke(messages)
            time.sleep(PACE)                       # pace the next call
            return out
        except Exception as exc:                   # noqa: BLE001 - we re-raise below
            text = str(exc).lower()
            if "429" in text or "rate" in text or "quota" in text:
                wait = pause * (2 ** attempt)
                print("  [rate limit] sleeping %.1fs (attempt %d/%d)" % (wait, attempt + 1, retries))
                time.sleep(wait)
                continue
            raise
    raise RuntimeError("rate limited after %d attempts" % retries)


# Published Groq list price for this model at the time of writing, in USD per
# 1M tokens. Substitute your own numbers - the METHOD is the lesson, not these
# two constants.
PRICE_IN_PER_M = 0.29
PRICE_OUT_PER_M = 0.59


def usd(in_tok: int, out_tok: int) -> float:
    """Convert a token count into dollars at the prices above."""
    return in_tok / 1e6 * PRICE_IN_PER_M + out_tok / 1e6 * PRICE_OUT_PER_M


print("env file :", ENV_PATH, "(exists:", ENV_PATH.exists(), ")")
print("model    :", GROQ_MODEL)
print("key      : loaded, %d chars" % len(GROQ_KEY))


### The embedding model


In [ ]:
# `all-MiniLM-L6-v2` is 22M parameters, runs on CPU, and encodes a short sentence
# in single-digit milliseconds. That speed is the whole point: routing has to be
# cheap enough that it is obviously worth doing before the expensive call.

from sentence_transformers import SentenceTransformer
import numpy as np

encoder = SentenceTransformer("all-MiniLM-L6-v2")


def embed(texts):
    """Encode a list of strings into L2-normalised vectors.

    Normalising means the dot product IS the cosine similarity, so every score
    below lives in [-1, 1] and is directly comparable.
    """
    return encoder.encode(list(texts), normalize_embeddings=True)


v = embed(["hello there", "hi!", "what is the capital of France"])
print("vector shape :", v.shape)
print("hello/hi     : %.3f" % float(v[0] @ v[1]))
print("hello/capital: %.3f" % float(v[0] @ v[2]))


### The route set


In [ ]:
# A ROUTE is: a name, a handful of EXAMPLE UTTERANCES, and a handler.
# Nothing more. There is no training step and no classifier to fit.

ROUTES = {
    "greeting": [
        "hi there",
        "hello",
        "hey, good morning",
        "yo",
        "good evening",
    ],
    "shipping_faq": [
        "how long does delivery take",
        "when will my parcel arrive",
        "do you ship internationally",
        "what are your shipping costs",
        "how fast is standard delivery",
    ],
    "order_status": [
        "where is order 1042",
        "track my order 88",
        "what happened to order number 7",
        "status of order 1042 please",
        "has order 88 shipped yet",
    ],
    "arithmetic": [
        "what is 18 times 24",
        "compute 145 plus 92",
        "calculate 900 divided by 12",
        "how much is 37 minus 19",
        "multiply 13 by 7",
    ],
}

ROUTE_NAMES = list(ROUTES)
print("%d routes, %d example utterances total"
      % (len(ROUTES), sum(len(v) for v in ROUTES.values())))


### 1. What do the scores look like?

Before choosing a threshold, look at the distribution. We score every held-out
utterance and separate the in-domain ones from the out-of-domain ones.

In [4]:
ROUTE_VECTORS = {n: embed(ex) for n, ex in ROUTES.items()}


def top_score(text):
    q = embed([text])[0]
    s = {n: float((v @ q).max()) for n, v in ROUTE_VECTORS.items()}
    best = max(s, key=s.get)
    return best, s[best]


IN_DOMAIN = [
    ("morning!", "greeting"), ("hiya", "greeting"), ("good afternoon to you", "greeting"),
    ("hey folks", "greeting"),
    ("do you deliver to Portugal", "shipping_faq"),
    ("how much do you charge for postage", "shipping_faq"),
    ("is next-day delivery available", "shipping_faq"),
    ("what's your delivery window", "shipping_faq"),
    ("any update on order 1042", "order_status"),
    ("I placed order 7 last week, where is it", "order_status"),
    ("check order number 88", "order_status"),
    ("did order 1042 leave the warehouse", "order_status"),
    ("what's 45 times 3", "arithmetic"), ("add 210 and 66 for me", "arithmetic"),
    ("divide 480 by 16", "arithmetic"), ("subtract 19 from 100", "arithmetic"),
]
OUT_OF_DOMAIN = [
    "who wrote Pride and Prejudice",
    "summarise the theory of plate tectonics",
    "write me a haiku about rain",
    "is my data used to train your models",
    "my cat has been sick since Tuesday, what should I do",
    "explain the difference between TCP and UDP",
    "recommend a good sourdough recipe",
    "what is your refund policy for damaged goods",
]

in_scores = [top_score(t)[1] for t, _ in IN_DOMAIN]
ood_scores = [top_score(t)[1] for t in OUT_OF_DOMAIN]

print("in-domain   n=%2d  min %.3f  median %.3f  max %.3f"
      % (len(in_scores), min(in_scores), float(np.median(in_scores)), max(in_scores)))
print("out-of-dom  n=%2d  min %.3f  median %.3f  max %.3f"
      % (len(ood_scores), min(ood_scores), float(np.median(ood_scores)), max(ood_scores)))
print()
print("overlap: %s" % ("YES - the distributions cross, so no threshold is perfect"
                       if max(ood_scores) >= min(in_scores) else "none on this sample"))

in-domain   n=16  min 0.319  median 0.609  max 0.898
out-of-dom  n= 8  min 0.029  median 0.121  max 0.237

overlap: none on this sample


In [5]:
# A text histogram - no plotting library needed to see the shape.
def hist(values, label, lo=0.0, hi=1.0, bins=20):
    counts = [0] * bins
    for v in values:
        idx = min(bins - 1, max(0, int((v - lo) / (hi - lo) * bins)))
        counts[idx] += 1
    print(label)
    for i, c in enumerate(counts):
        edge = lo + (hi - lo) * i / bins
        print("  %.2f | %s %s" % (edge, "#" * c, c or ""))


hist(in_scores, "IN-DOMAIN top scores")
print()
hist(ood_scores, "OUT-OF-DOMAIN top scores")

IN-DOMAIN top scores
  0.00 |  
  0.05 |  
  0.10 |  
  0.15 |  
  0.20 |  
  0.25 |  
  0.30 | # 1
  0.35 |  
  0.40 |  
  0.45 | ## 2
  0.50 | ## 2
  0.55 | ### 3
  0.60 | ## 2
  0.65 | ## 2
  0.70 | ## 2
  0.75 |  
  0.80 | # 1
  0.85 | # 1
  0.90 |  
  0.95 |  

OUT-OF-DOMAIN top scores
  0.00 | ## 2
  0.05 | # 1
  0.10 | ### 3
  0.15 |  
  0.20 | ## 2
  0.25 |  
  0.30 |  
  0.35 |  
  0.40 |  
  0.45 |  
  0.50 |  
  0.55 |  
  0.60 |  
  0.65 |  
  0.70 |  
  0.75 |  
  0.80 |  
  0.85 |  
  0.90 |  
  0.95 |  


The out-of-domain scores are not near zero. Cosine similarity between two ordinary
English sentences is rarely below ~0.1 even when they share nothing, and a question
about a refund policy genuinely *is* somewhat similar to a shipping FAQ. **Your
threshold lives in the middle of a crowded range, not near zero.**

### 2. The threshold sweep

Two things move in opposite directions as you raise the threshold:

- **coverage** - the fraction of traffic the router handles itself (your saving)
- **routed accuracy** - of the traffic it does handle, how much it got right

Plus the one that actually costs you: **misroute rate** - out-of-domain traffic
accepted into a route.

In [6]:
ALL = [(t, g) for t, g in IN_DOMAIN] + [(t, None) for t in OUT_OF_DOMAIN]
PRE = [(t, g, *top_score(t)) for t, g in ALL]      # score everything once

print("%-8s %-9s %-9s %-9s %-9s" % ("thresh", "coverage", "routed_acc", "misroute", "overall"))
print("-" * 48)
sweep = []
for th in [round(x * 0.05, 2) for x in range(4, 17)]:
    routed = missed = misrouted = correct_routed = 0
    overall_correct = 0
    for text, gold, best, sc in PRE:
        pred = best if sc >= th else None
        if pred is not None:
            routed += 1
            if pred == gold:
                correct_routed += 1
            elif gold is None:
                misrouted += 1
        else:
            if gold is not None:
                missed += 1
        overall_correct += (pred == gold)
    cov = routed / len(PRE)
    racc = correct_routed / routed if routed else float("nan")
    mis = misrouted / len(OUT_OF_DOMAIN)
    print("%-8.2f %-9.3f %-9.3f %-9.3f %-9.3f"
          % (th, cov, racc, mis, overall_correct / len(PRE)))
    sweep.append((th, cov, racc, mis, overall_correct / len(PRE)))

thresh   coverage  routed_acc misroute  overall  
------------------------------------------------
0.20     0.750     0.889     0.250     0.917    
0.25     0.667     1.000     0.000     1.000    
0.30     0.667     1.000     0.000     1.000    
0.35     0.625     1.000     0.000     0.958    
0.40     0.625     1.000     0.000     0.958    
0.45     0.625     1.000     0.000     0.958    
0.50     0.542     1.000     0.000     0.875    
0.55     0.458     1.000     0.000     0.792    
0.60     0.333     1.000     0.000     0.667    
0.65     0.250     1.000     0.000     0.583    
0.70     0.167     1.000     0.000     0.500    
0.75     0.083     1.000     0.000     0.417    
0.80     0.083     1.000     0.000     0.417    


In [7]:
best_row = max(sweep, key=lambda r: r[4])
print("best OVERALL accuracy at threshold %.2f (%.3f)" % (best_row[0], best_row[4]))

# But overall accuracy weights both error types equally, and they are NOT equal.
# Cost-weighted choice: a misroute is (say) 5x worse than an unnecessary fallback.
MISROUTE_PENALTY = 5.0
print()
print("%-8s %-10s %-10s %-10s" % ("thresh", "missed", "misroutes", "weighted_loss"))
print("-" * 42)
scored = []
for th, cov, racc, mis, overall in sweep:
    missed = sum(1 for t, g, b, s in PRE if g is not None and s < th)
    misroutes = sum(1 for t, g, b, s in PRE if g is None and s >= th)
    wrong_route = sum(1 for t, g, b, s in PRE if g is not None and s >= th and b != g)
    loss = missed * 1.0 + (misroutes + wrong_route) * MISROUTE_PENALTY
    scored.append((th, loss))
    print("%-8.2f %-10d %-10d %-10.1f" % (th, missed, misroutes + wrong_route, loss))

CHOSEN = min(scored, key=lambda r: r[1])[0]
print()
print("CHOSEN THRESHOLD = %.2f (lowest cost-weighted loss at %dx misroute penalty)"
      % (CHOSEN, MISROUTE_PENALTY))

best OVERALL accuracy at threshold 0.25 (1.000)

thresh   missed     misroutes  weighted_loss
------------------------------------------
0.20     0          2          10.0      
0.25     0          0          0.0       
0.30     0          0          0.0       
0.35     1          0          1.0       
0.40     1          0          1.0       
0.45     1          0          1.0       
0.50     3          0          3.0       
0.55     5          0          5.0       
0.60     8          0          8.0       
0.65     10         0          10.0      
0.70     12         0          12.0      
0.75     14         0          14.0      
0.80     14         0          14.0      

CHOSEN THRESHOLD = 0.25 (lowest cost-weighted loss at 5x misroute penalty)


### Why the penalty is not 1

A **missed route** costs you one model call you did not need. On this model that is
a fraction of a cent, and the user still gets a correct answer.

A **misroute** sends "what is your refund policy for damaged goods" to the shipping
FAQ handler, which confidently returns the wrong stored answer. The user does not
know it is wrong. That is a support ticket, or worse.

The two are not the same size of mistake, so optimising plain accuracy - which
treats them as equal - gives you the wrong threshold. Put a number on the asymmetry
and optimise that instead. The number is a business decision, not an ML one.

### 3. The fallback route

The fallback is not an `else`. It is the route that handles everything you did not
anticipate, which on day one is most of your traffic. Give it the same attention as
the others:

- It gets the **full-strength model** and the **full context**, because the cheap
  paths are gone.
- Its traffic is **logged**, because it is your source of new routes.
- It should be able to **say it does not know**, since it now receives genuinely
  off-topic requests.

Here is the whole router, fallback included.

In [8]:
import re

ORDER_RE = re.compile(r"order\s*(?:number\s*|#\s*|no\.?\s*)?(\d+)", re.I)
ORDERS = {"7": "delivered 12 March", "88": "in transit, arriving Thursday",
          "1042": "packed, leaves the warehouse tonight"}

FALLBACK_LOG = []


def handle_greeting(text):
    return "Hello! How can I help with your order today?"


def handle_shipping_faq(text):
    return ("Standard delivery is 3-5 working days within the EU and 7-10 days "
            "internationally. Shipping is free over EUR 40.")


def handle_order_status(text):
    m = ORDER_RE.search(text)
    if not m:
        return None                       # handler REFUSES -> falls back
    return "Order %s: %s." % (m.group(1), ORDERS.get(m.group(1), "no record found"))


def handle_arithmetic(text):
    nums = [int(n) for n in re.findall(r"-?\d+", text)]
    if len(nums) != 2:
        return None                       # handler REFUSES -> falls back
    a, b = nums
    if any(w in text.lower() for w in ("times", "multiply", "product")):
        return "%d x %d = %d" % (a, b, a * b)
    if any(w in text.lower() for w in ("plus", "add", "sum")):
        return "%d + %d = %d" % (a, b, a + b)
    if any(w in text.lower() for w in ("divided", "divide")):
        return "%d / %d = %.4g" % (a, b, a / b)
    if any(w in text.lower() for w in ("minus", "subtract")):
        return "%d - %d = %d" % (a, b, a - b) if "from" not in text.lower() else "%d - %d = %d" % (b, a, b - a)
    return None


HANDLERS = {"greeting": handle_greeting, "shipping_faq": handle_shipping_faq,
            "order_status": handle_order_status, "arithmetic": handle_arithmetic}

fallback_llm = make_llm(temperature=0.0, max_tokens=160)


def fallback(text, reason):
    """THE FALLBACK ROUTE. Full model, full context, and it is allowed to decline."""
    FALLBACK_LOG.append((text, reason))
    msg = safe_invoke(fallback_llm, [
        ("system", "You are a support assistant for an online shop. Answer in at "
                   "most two sentences. If the question is outside the shop's "
                   "domain, say so plainly rather than guessing."),
        ("user", text)])
    return msg.content.strip(), (msg.usage_metadata or {})


def answer(text, threshold=CHOSEN):
    best, sc = top_score(text)
    if sc < threshold:
        return fallback(text, "low confidence %.3f" % sc)[0], "fallback(low-conf)", 0, 0
    result = HANDLERS[best](text)
    if result is None:
        return fallback(text, "handler %s declined" % best)[0], "fallback(refused)", 0, 0
    return result, best, 0, 0


print("router ready. threshold = %.2f" % CHOSEN)

router ready. threshold = 0.25


In [9]:
DEMO = [
    "morning!",
    "check order number 88",
    "what's 45 times 3",
    "do you deliver to Portugal",
    "where is my stuff",                       # order intent, no number -> handler refuses
    "recommend a good sourdough recipe",       # out of domain -> low confidence
]

for d in DEMO:
    out, taken, _, _ = answer(d)
    print("%-42s [%s]" % (d, taken))
    print("    %s" % out.replace("\n", " ")[:140])
    print()

print("fallback log (%d entries) - this is your route backlog:" % len(FALLBACK_LOG))
for text, reason in FALLBACK_LOG:
    print("  %-45s %s" % (text[:45], reason))

morning!                                   [greeting]
    Hello! How can I help with your order today?

check order number 88                      [order_status]
    Order 88: in transit, arriving Thursday.

what's 45 times 3                          [arithmetic]
    45 x 3 = 135

do you deliver to Portugal                 [shipping_faq]
    Standard delivery is 3-5 working days within the EU and 7-10 days internationally. Shipping is free over EUR 40.

where is my stuff                          [shipping_faq]
    Standard delivery is 3-5 working days within the EU and 7-10 days internationally. Shipping is free over EUR 40.



recommend a good sourdough recipe          [fallback(low-conf)]
    I cannot provide recipe recommendations as I am an online shop support assistant. Please consult a culinary resource for baking advice.

fallback log (1 entries) - this is your route backlog:
  recommend a good sourdough recipe             low confidence 0.142


Two different fallback reasons showed up, and they mean different things:

- `low confidence` - the router does not recognise this at all. If a *cluster* of
  these appears in your logs, that cluster is a route you have not written yet.
- `handler declined` - the router was right about the topic but the handler could
  not complete. "Where is my stuff" is genuinely `order_status`; it just lacks an
  order number. The fix is not a router change, it is a handler that asks a
  follow-up question.

Distinguishing these two in your logs is worth the ten lines it costs.

### Pitfalls

- **A threshold copied from a blog post.** Scores are encoder-specific. Re-measure
  when you change the embedding model, and re-measure after a route-set edit.
- **One global threshold for every route.** Routes with tight, distinctive examples
  (arithmetic) can afford a lower threshold than fuzzy ones (FAQ). Per-route
  thresholds are a cheap upgrade.
- **A fallback that is weaker than the routes.** If your fallback is a smaller model,
  you have made the hardest traffic get the worst treatment.
- **No logging on the fallback.** Then you never learn what routes you are missing.

### Next

Notebook 04 puts a real workload through this and measures what it saved - tokens,
dollars and latency - against always calling the model.